# 임베딩·벡터·내적·역전파(Backprop) 기초 수학: 직관 + 실습

이 노트북은 아래를 **수학/직관 → 코드** 순서로 연결합니다.

- 벡터/행렬, 노름(norm), 내적(dot), 코사인 유사도
- 선형변환(가중치 행렬 W)과 표현공간(representation space)
- 임베딩(embedding)의 개념: “의미를 좌표로 바꾸는 것”
- 손실함수(loss)와 기울기(gradient), 경사하강법(GD)
- 역전파(backpropagation): 연쇄법칙으로 기울기를 뒤로 전달

> 실행 환경: Python 3


In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)


## 1) 벡터는 “특징의 리스트”다

### 벡터(Vector)
- 길이가 `d`인 숫자 배열: \(x \in \mathbb{R}^d\)
- 해석: 각 차원은 **특징(feature)**, 벡터는 **특징의 묶음**

예: 문장/단어/이미지/사용자 행동 등을 숫자 특징으로 만든 것이 벡터.


In [ ]:
# 벡터 예시
x = np.array([2.0, -1.0, 0.5])
y = np.array([-1.0, 2.0, 1.0])

x, y, x.shape


### 노름(Norm): 벡터의 “크기/길이”

가장 많이 쓰는 L2 노름:
\[
\|x\|_2 = \sqrt{\sum_i x_i^2}
\]


In [ ]:
def l2_norm(v):
    return float(np.sqrt(np.sum(v*v)))

l2_norm(x), l2_norm(y)


## 2) 내적(Dot product)은 “정렬(alignment)”을 측정한다

\[
x \cdot y = \sum_i x_i y_i
\]

- 두 벡터가 같은 방향이면 내적이 큼(양수)
- 반대 방향이면 내적이 작음(음수)
- 직교(orthogonal)면 0 근처

### 코사인 유사도(Cosine similarity)
크기에 영향받지 않고 **방향**만 비교:
\[
\cos(x,y) = \frac{x\cdot y}{\|x\|\|y\|}
\]
임베딩 검색/추천에서 자주 사용.


In [ ]:
def dot(a,b): return float(np.dot(a,b))
def cosine_sim(a,b):
    return dot(a,b) / (l2_norm(a)*l2_norm(b) + 1e-12)

dot(x,y), cosine_sim(x,y)


## 3) 가중치(Weights)는 “선형변환”이다

신경망에서 가장 기본 블록:
\[
z = W x + b
\]
- \(W\): 가중치 행렬 (입력을 다른 공간으로 “회전/스케일/혼합”)
- \(b\): 편향 (평행이동)

직관:
- **W는 특징을 섞어서 새로운 특징을 만든다**
- 임베딩 레이어도 결국 “학습되는 W”로 볼 수 있다


In [ ]:
# 간단한 선형변환
W = np.array([[1.0, 0.5, -0.5],
              [0.0, 1.0,  1.0]])
b = np.array([0.1, -0.2])

z = W @ x + b
z


## 4) 임베딩(Embedding)이란?

임베딩은 대상을 **연속 벡터 공간의 점**으로 매핑하는 것.

- 단어 임베딩: 단어ID → 벡터
- 문장 임베딩: 문장 → 벡터
- 사용자/아이템 임베딩: 추천 시스템

### 왜 유용한가?
- “비슷한 의미”의 대상은 **가까운 좌표**를 갖도록 학습 가능
- 검색/추천/클러스터링/분류 등에서 거리/내적만으로 유사도를 계산

### 임베딩 레이어의 수학적 형태 (lookup)
단어ID가 `i`일 때:
- 임베딩 행렬 \(E \in \mathbb{R}^{V \times d}\)
- 임베딩 벡터는 \(e_i = E[i]\) (i번째 행을 꺼냄)

즉, 임베딩은 “거대한 표(테이블)에서 한 줄을 뽑는 것”.


In [ ]:
# 작은 어휘(V=6), 임베딩 차원(d=4) 예시
V, d = 6, 4
rng = np.random.default_rng(0)
E = rng.normal(size=(V, d))

tokens = ["king","queen","man","woman","apple","orange"]
token_to_id = {t:i for i,t in enumerate(tokens)}

def embed(token):
    return E[token_to_id[token]]

pd.DataFrame(E, index=tokens, columns=[f"dim{j}" for j in range(d)]).round(4)


### 임베딩 유사도 예시
학습 전 랜덤이므로 의미가 없지만 “계산 방식”을 보여줍니다.


In [ ]:
pairs = [("king","queen"), ("apple","orange"), ("king","apple")]
rows = []
for a,b in pairs:
    rows.append({
        "a": a, "b": b,
        "dot": dot(embed(a), embed(b)),
        "cosine": cosine_sim(embed(a), embed(b))
    })
pd.DataFrame(rows).round(4)


## 5) 손실함수(Loss)와 기울기(Gradient)

학습은 “정답에 가까워지도록 W를 조정”하는 과정.

- 모델 출력 \(\hat{y}\)
- 정답 \(y\)
- 손실 \(L(\hat{y}, y)\): 틀린 정도를 숫자로

가중치를 업데이트:
\[
W \leftarrow W - \eta \nabla_W L
\]
- \(\nabla_W L\): 손실을 W로 미분한 것(기울기)
- \(\eta\): 학습률(learning rate)


## 6) Backpropagation(역전파)의 핵심: 연쇄법칙(Chain rule)

간단한 2단 계산을 생각해봅시다.
\[
u = f(x),\quad L = g(u)
\]
그럼
\[
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial u}\frac{\partial u}{\partial x}
\]
역전파는 이걸 **그래프를 따라 뒤로** 곱해가며 구하는 것.

아래는 가장 기본적인 1층 선형회귀:
\[
\hat{y}=Wx + b
\]
\[
L = \frac{1}{2}(\hat{y}-y)^2
\]
이때
\[
\frac{\partial L}{\partial W} = (\hat{y}-y) x^T
\]
\[
\frac{\partial L}{\partial b} = (\hat{y}-y)
\]


In [ ]:
# 1D 입력->1D 출력 선형회귀: y_hat = w*x + b
# 손실: 0.5*(y_hat - y)^2

def forward(w, b, x):
    return w*x + b

def loss(y_hat, y):
    return 0.5*(y_hat - y)**2

def grads(w, b, x, y):
    y_hat = forward(w,b,x)
    dL_dyhat = (y_hat - y)        # derivative of 0.5*(y_hat-y)^2 w.r.t y_hat
    dL_dw = dL_dyhat * x
    dL_db = dL_dyhat * 1.0
    return float(dL_dw), float(dL_db), float(y_hat), float(loss(y_hat,y))

# 데이터(선형 관계 y=2x+1 + noise)
rng = np.random.default_rng(1)
xs = rng.uniform(-2,2,size=200)
ys = 2*xs + 1 + rng.normal(scale=0.3,size=200)

w, b = 0.0, 0.0
lr = 0.05

history = []
for step in range(200):
    # 배치 평균 기울기
    dws, dbs, Ls = [], [], []
    for x,y in zip(xs,ys):
        dL_dw, dL_db, y_hat, L = grads(w,b,x,y)
        dws.append(dL_dw); dbs.append(dL_db); Ls.append(L)
    dW = np.mean(dws)
    dB = np.mean(dbs)
    w -= lr*dW
    b -= lr*dB
    history.append((step, w, b, float(np.mean(Ls))))

pd.DataFrame(history, columns=["step","w","b","mean_loss"]).tail().round(4)


## 7) 임베딩을 “학습”하는 최소 예시: Skip-gram 스타일(매우 단순화)

목표:
- (중심 단어, 주변 단어) 쌍이 자주 나오면 **내적이 커지도록** 학습
- 자주 함께 나오지 않으면 내적이 작아지도록

여기서는 교육용으로:
- 중심 임베딩: \(E\) (V×d)
- 주변 임베딩: \(C\) (V×d)
- 점수: \(s = e_{center} \cdot c_{context}\)
- 확률(시그모이드): \(p=\sigma(s)\)
- 라벨: 긍정쌍=1, 부정쌍=0
- 손실(로지스틱): \(L= -[y\log p + (1-y)\log(1-p)]\)

핵심: 내적을 통해 “가깝게/멀게” 만든다.


In [ ]:
def sigmoid(t):
    return 1/(1+np.exp(-t))

# toy corpus
corpus = [
    ["king","queen","king","man"],
    ["queen","woman","queen","king"],
    ["apple","orange","apple","orange"],
    ["man","woman","man","king"],
]

# build pairs (center, context) with window=1
pairs = []
for sent in corpus:
    for i,center in enumerate(sent):
        for j in [i-1, i+1]:
            if 0 <= j < len(sent):
                pairs.append((center, sent[j]))

pairs[:10], len(pairs)


In [ ]:
# initialize embeddings
V = len(tokens)
d = 8
rng = np.random.default_rng(42)
E = rng.normal(scale=0.3, size=(V,d))  # center
C = rng.normal(scale=0.3, size=(V,d))  # context

def train_step(center_id, context_id, label, lr=0.3):
    # score
    e = E[center_id]      # (d,)
    c = C[context_id]     # (d,)
    s = float(np.dot(e,c))
    p = float(sigmoid(s))
    # loss
    L = -(label*np.log(p+1e-12) + (1-label)*np.log(1-p+1e-12))
    # gradients
    # dL/ds = p - label
    dL_ds = (p - label)
    # dL/de = dL/ds * ds/de = (p-label) * c
    # dL/dc = (p-label) * e
    grad_e = dL_ds * c
    grad_c = dL_ds * e
    # update
    E[center_id] -= lr*grad_e
    C[context_id] -= lr*grad_c
    return L, p, s

# negative sampling helper
all_ids = list(range(V))
def sample_negative(exclude_id):
    nid = exclude_id
    while nid == exclude_id:
        nid = random.choice(all_ids)
    return nid

# training
random.seed(0)
lr = 0.3
losses = []
for epoch in range(200):
    random.shuffle(pairs)
    total = 0.0
    for center, context in pairs:
        cid = token_to_id[center]
        pos_id = token_to_id[context]
        # positive
        L, p, s = train_step(cid, pos_id, 1, lr=lr)
        total += L
        # one negative
        neg_id = sample_negative(pos_id)
        L, p, s = train_step(cid, neg_id, 0, lr=lr)
        total += L
    losses.append(total/ (2*len(pairs)))

pd.DataFrame({"epoch": np.arange(len(losses)), "loss": losses}).tail()


In [ ]:
# after training, check similarities
def cos_from_E(a,b):
    return cosine_sim(E[token_to_id[a]], E[token_to_id[b]])

tests = [("king","queen"), ("man","woman"), ("apple","orange"), ("king","apple")]
pd.DataFrame([{"a":a,"b":b,"cosine(E)": cos_from_E(a,b)} for a,b in tests]).round(4)


## 8) pandas로 “임베딩 테이블” 보기 + 최근접 이웃(Nearest Neighbors)

실무에서 임베딩은 대개:
- (id → vector) 테이블로 저장 (예: Parquet)
- 검색 시: 쿼리 벡터와 DB의 벡터들을 cosine/dot으로 비교해 top-k 반환


In [ ]:
emb_df = pd.DataFrame(E, index=tokens, columns=[f"d{j}" for j in range(d)])
emb_df.head().round(4)


In [ ]:
def top_k_neighbors(query_token, k=3):
    q = E[token_to_id[query_token]]
    sims = []
    for t in tokens:
        sims.append((t, cosine_sim(q, E[token_to_id[t]])))
    sims.sort(key=lambda x: x[1], reverse=True)
    return pd.DataFrame(sims[:k+1], columns=["token","cosine"]).round(4)

top_k_neighbors("king", k=5)


## 9) 다음 학습 로드맵(권장 순서)

1. **선형대수**
   - 벡터/행렬, 내적, 직교/기저, 고유값(선택)
2. **미분/미적분**
   - 편미분, 기울기, 연쇄법칙
3. **최적화**
   - 경사하강법, 모멘텀/Adam(개념)
4. **확률/정보이론(기초)**
   - 로지스틱(시그모이드), 크로스엔트로피
5. **임베딩 모델**
   - word2vec/negative sampling, matrix factorization, transformer embedding
6. **실전**
   - cosine vs dot, 정규화, ANN(근사 최근접 탐색: FAISS 등)

이 노트북은 1~3, 5의 최소 골격을 직접 구현해봤다는 점이 포인트입니다.
